# CNN Training – Brain Tumor MRI Classification

**Dataset:** Brain Tumor MRI Dataset (Kaggle)  
**Classes:** Glioma, Meningioma, No Tumor, Pituitary  
**Model:** Convolutional Neural Network (TensorFlow/Keras)

---

## Step 1 – Download Dataset via Kaggle CLI

Run this in your terminal (with venv activated):

```bash
pip install kaggle
```

Place your `kaggle.json` API key in:
- Windows: `C:\Users\<YourName>\.kaggle\kaggle.json`

Then run:
```bash
kaggle datasets download -d masoudnickparvar/brain-tumor-mri-dataset -p datasets/brain_tumor --unzip
```

In [ ]:
import os, sys, json, shutil
import numpy as np
import matplotlib.pyplot as plt

# Add project root to path
sys.path.insert(0, os.path.abspath('..'))

import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import classification_report, confusion_matrix
import plotly.express as px
import plotly.graph_objects as go

print('TensorFlow:', tf.__version__)
print('GPU available:', len(tf.config.list_physical_devices('GPU')) > 0)

In [ ]:
# ── Configuration ──────────────────────────────────────────────
DATA_DIR      = '../datasets/brain_tumor'
MODEL_PATH    = '../saved_models/cnn_model.keras'
METRICS_PATH  = '../saved_models/cnn_metrics.json'
IMG_SIZE      = (150, 150)
BATCH_SIZE    = 32
EPOCHS        = 25
CLASSES       = ['glioma', 'meningioma', 'notumor', 'pituitary']
CLASS_LABELS  = ['Glioma', 'Meningioma', 'No Tumor', 'Pituitary']

# Locate Training folder (handles both flat and nested structures)
TRAIN_DIR = None
TEST_DIR  = None

for root, dirs, files in os.walk(DATA_DIR):
    if 'Training' in dirs:
        TRAIN_DIR = os.path.join(root, 'Training')
        TEST_DIR  = os.path.join(root, 'Testing')
        break

if not TRAIN_DIR:
    # Try flat structure
    TRAIN_DIR = os.path.join(DATA_DIR, 'Training')
    TEST_DIR  = os.path.join(DATA_DIR, 'Testing')

print('Train dir:', TRAIN_DIR)
print('Test dir :', TEST_DIR)
print('Exists   :', os.path.exists(TRAIN_DIR))

In [ ]:
# ── Dataset counts ─────────────────────────────────────────────
for split, d in [('Train', TRAIN_DIR), ('Test', TEST_DIR)]:
    if os.path.exists(d):
        for cls in sorted(os.listdir(d)):
            cls_path = os.path.join(d, cls)
            if os.path.isdir(cls_path):
                count = len(os.listdir(cls_path))
                print(f'{split}/{cls}: {count} images')

In [ ]:
# ── Data Generators ────────────────────────────────────────────
train_datagen = keras.preprocessing.image.ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.1,
    validation_split=0.15
)

test_datagen = keras.preprocessing.image.ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', subset='training', seed=42
)
val_gen = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', subset='validation', seed=42
)
test_gen = test_datagen.flow_from_directory(
    TEST_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=False
)

print('Class indices:', train_gen.class_indices)
print(f'Train: {train_gen.samples} | Val: {val_gen.samples} | Test: {test_gen.samples}')

In [ ]:
# ── Build CNN Model ────────────────────────────────────────────
model = keras.Sequential([
    keras.layers.Input(shape=(*IMG_SIZE, 3)),

    keras.layers.Conv2D(32, (3,3), activation='relu', padding='same'),
    keras.layers.BatchNormalization(),
    keras.layers.MaxPooling2D(2,2),

    keras.layers.Conv2D(64, (3,3), activation='relu', padding='same'),
    keras.layers.BatchNormalization(),
    keras.layers.MaxPooling2D(2,2),

    keras.layers.Conv2D(128, (3,3), activation='relu', padding='same'),
    keras.layers.BatchNormalization(),
    keras.layers.MaxPooling2D(2,2),

    keras.layers.Conv2D(256, (3,3), activation='relu', padding='same'),
    keras.layers.MaxPooling2D(2,2),

    keras.layers.Flatten(),
    keras.layers.Dense(512, activation='relu'),
    keras.layers.Dropout(0.5),
    keras.layers.Dense(4, activation='softmax')
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
# ── Callbacks ──────────────────────────────────────────────────
callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6),
    keras.callbacks.ModelCheckpoint(MODEL_PATH, monitor='val_accuracy', save_best_only=True)
]

# ── Train ──────────────────────────────────────────────────────
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    callbacks=callbacks
)

print('\nTraining complete!')

In [ ]:
# ── Evaluate on Test Set ───────────────────────────────────────
test_loss, test_acc = model.evaluate(test_gen, verbose=0)
print(f'Test Accuracy : {test_acc*100:.2f}%')
print(f'Test Loss     : {test_loss:.4f}')

# Predictions
test_gen.reset()
y_pred_probs = model.predict(test_gen, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = test_gen.classes

report = classification_report(y_true, y_pred, target_names=CLASS_LABELS, output_dict=True)
print('\nClassification Report:')
print(classification_report(y_true, y_pred, target_names=CLASS_LABELS))

In [ ]:
# ── Confusion Matrix ───────────────────────────────────────────
cm = confusion_matrix(y_true, y_pred)
fig = px.imshow(cm, text_auto=True, color_continuous_scale='Blues',
                x=CLASS_LABELS, y=CLASS_LABELS,
                labels=dict(x='Predicted', y='Actual'),
                title='Confusion Matrix')
fig.update_layout(template='plotly_white')
fig.show()

In [ ]:
# ── Training Curves ────────────────────────────────────────────
epochs_range = list(range(1, len(history.history['accuracy']) + 1))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(epochs_range, history.history['accuracy'], label='Train')
axes[0].plot(epochs_range, history.history['val_accuracy'], label='Validation', linestyle='--')
axes[0].set_title('Accuracy'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, history.history['loss'], label='Train', color='red')
axes[1].plot(epochs_range, history.history['val_loss'], label='Validation', linestyle='--', color='orange')
axes[1].set_title('Loss'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ── Save Metrics for Flask App ─────────────────────────────────
metrics = {
    'accuracy'  : round(float(test_acc), 4),
    'precision' : round(float(report['weighted avg']['precision']), 4),
    'recall'    : round(float(report['weighted avg']['recall']), 4),
    'f1'        : round(float(report['weighted avg']['f1-score']), 4),
    'train_acc' : [round(float(v), 4) for v in history.history['accuracy']],
    'val_acc'   : [round(float(v), 4) for v in history.history['val_accuracy']],
    'train_loss': [round(float(v), 4) for v in history.history['loss']],
    'val_loss'  : [round(float(v), 4) for v in history.history['val_loss']],
    'classes'   : CLASS_LABELS,
    'cm'        : cm.tolist()
}

with open(METRICS_PATH, 'w') as f:
    json.dump(metrics, f, indent=2)

print(f'Model saved  : {MODEL_PATH}')
print(f'Metrics saved: {METRICS_PATH}')
print(f'\nFinal Accuracy : {metrics["accuracy"]*100:.1f}%')
print(f'F1 Score       : {metrics["f1"]*100:.1f}%')

## Done!

The trained model is saved to `saved_models/cnn_model.keras`.  
Restart the Flask app and go to **CNN Classifier** to start classifying Brain MRI images.